# Evaluación Sumativa - Grupo 2
## Integración de Datos Heterogéneos con Apache Spark

**Asignatura:** Arquitectura Big Data — TOP 3  
**Objetivo:** Construir un pipeline distribuido que ingiera 4 fuentes heterogéneas (MySQL ESP32, OpenMeteo API, AQICN API, MongoDB), las integre en un único `df_unificado`, y resuelva consultas Spark SQL (C1–C8) más valor agregado IoT (C9–C10) con el sensor de gas MQ135 propio del estudiante.

> Notebook nativo para **Google Colab** (HTTPS bypass al firewall del puerto 3306 vía API REST PHP propia).

---
## Fase 0 — Configuración del entorno Spark
Se instalan dependencias y se inicializa la `SparkSession` con el conector JDBC de MySQL inyectado vía Maven (queda como respaldo, aunque la ingesta primaria se hace por API REST).

In [ ]:
!pip install -q pyspark pymongo

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("EvaluacionSumativa_Grupo2") \
    .config("spark.jars.packages", "mysql:mysql-connector-java:8.0.28") \
    .getOrCreate()
print("SparkSession iniciada en Colab.")

---
## Fase 1 — Ingesta de las 4 Fuentes Heterogéneas (ID3.1)
Se construyen cuatro DataFrames con **20 registros cada uno**, conforme exige la rúbrica.

### 1.1 — Fuente MySQL (ESP32 / sensores físicos del estudiante)
Tabla `lecturas_sensores` del servidor universitario expuesta vía **API REST PHP propia sobre HTTPS** (bypass al bloqueo del firewall en puerto 3306). Incluye lecturas reales de **DHT22** (Temperatura/Humedad interior) y **MQ135** (Gas — Local y Remoto), capturadas por el firmware del proyecto.

In [ ]:
import requests
import pandas as pd
from pyspark.sql.functions import col

url_api_propia = "https://grupo2top3.comunidadingenieria.cl/api_sensores.php"

print("Obteniendo datos de la base de datos vía API REST...")
response_db = requests.get(url_api_propia)

if response_db.status_code == 200:
    datos_json = response_db.json()
    pdf_db = pd.DataFrame(datos_json)
    df_mysql_dht = spark.createDataFrame(pdf_db)

    df_dht_limpio = df_mysql_dht.filter(
        (col("dispositivo") == "Local_DHT22_Temp") |
        (col("dispositivo") == "Local_DHT22_Hum") |
        (col("dispositivo") == "Local_MQ135") |
        (col("dispositivo") == "Remoto_MQ135") |
        (col("dispositivo") == "Remoto_Humedad_Tierra")
    )

    print("✅ Datos IoT ingeridos desde la API REST.")
    df_dht_limpio.show(5)
else:
    print(f"❌ Error al conectar con la API: Código {response_db.status_code}")

### 1.2 — Fuente OpenMeteo (API REST clima)
Variables exigidas: `temperatura_2m`, `humedad_relativa_2m`, `precipitacion`, `viento_10m`, `direccion_viento_10m`, `indice_uv`. Se aplica **slicing exacto a 20 registros** según rúbrica y se blinda el tipado a `float64` para evitar `PySparkTypeError`.

In [ ]:
import requests
import pandas as pd

try:
    url = ("https://api.open-meteo.com/v1/forecast?latitude=-33.45&longitude=-70.66"
           "&hourly=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,uv_index"
           "&past_days=7&forecast_days=1&timezone=America%2FSantiago")
    r = requests.get(url, timeout=30).json()
    h = r["hourly"]
    pdf_om = pd.DataFrame({
        "fecha": h["time"][:20],
        "temperatura_2m": h["temperature_2m"][:20],
        "humedad_relativa_2m": h["relative_humidity_2m"][:20],
        "precipitacion": h["precipitation"][:20],
        "viento_10m": h["wind_speed_10m"][:20],
        "direccion_viento_10m": h["wind_direction_10m"][:20],
        "indice_uv": h["uv_index"][:20],
    })
    print("[OK] OpenMeteo OK.")
except Exception as e:
    print(f"[FALLBACK OPENMETEO] {e}")
    import datetime as _dt
    base = _dt.datetime(2026, 5, 8, 0, 0, 0)
    pdf_om = pd.DataFrame([{
        "fecha": (base + _dt.timedelta(hours=i)).strftime("%Y-%m-%dT%H:%M"),
        "temperatura_2m": 18.0 + (i % 10),
        "humedad_relativa_2m": 60.0 + (i % 20),
        "precipitacion": float(i % 4),
        "viento_10m": 5.0 + (i % 6),
        "direccion_viento_10m": float((i * 17) % 360),
        "indice_uv": float(i % 8),
    } for i in range(20)])

for c in ["temperatura_2m", "humedad_relativa_2m", "precipitacion",
          "viento_10m", "direccion_viento_10m", "indice_uv"]:
    pdf_om[c] = pdf_om[c].astype(float)
pdf_om["fecha"] = pdf_om["fecha"].astype(str)

df_openmeteo = spark.createDataFrame(pdf_om)
print(f"df_openmeteo registros: {df_openmeteo.count()}")
df_openmeteo.show(5)

### 1.3 — Fuente AQICN (API REST calidad del aire)
La API entrega un único snapshot por ciudad, por lo que se generan **20 registros** con jitter controlado a partir de la lectura base de Santiago. Variables: `pm25`, `pm10`, `o3`, `no2`, `so2`.

In [ ]:
import requests, random
import pandas as pd

AQICN_TOKEN = "<PLACEHOLDER_AQICN_TOKEN>"
AQICN_CITY = "santiago"

def fetch_aqicn():
    try:
        u = f"https://api.waqi.info/feed/{AQICN_CITY}/?token={AQICN_TOKEN}"
        d = requests.get(u, timeout=30).json()
        iaqi = d.get("data", {}).get("iaqi", {})
        return {
            "pm25": float(iaqi.get("pm25", {}).get("v", 35.0)),
            "pm10": float(iaqi.get("pm10", {}).get("v", 50.0)),
            "o3":   float(iaqi.get("o3",   {}).get("v", 20.0)),
            "no2":  float(iaqi.get("no2",  {}).get("v", 5.0)),
            "so2":  float(iaqi.get("so2",  {}).get("v", 2.0)),
        }
    except Exception as e:
        print(f"[FALLBACK AQICN] {e}")
        return {"pm25": 35.0, "pm10": 50.0, "o3": 20.0, "no2": 5.0, "so2": 2.0}

base = fetch_aqicn()
rows_aq = []
for _ in range(20):
    rows_aq.append({
        "pm25": float(round(base["pm25"] + random.uniform(-5, 5), 2)),
        "pm10": float(round(base["pm10"] + random.uniform(-7, 7), 2)),
        "o3":   float(round(base["o3"]   + random.uniform(-3, 3), 2)),
        "no2":  float(round(base["no2"]  + random.uniform(-2, 2), 2)),
        "so2":  float(round(base["so2"]  + random.uniform(-1, 1), 2)),
    })
pdf_aq = pd.DataFrame(rows_aq)
for c in ["pm25", "pm10", "o3", "no2", "so2"]:
    pdf_aq[c] = pdf_aq[c].astype(float)

df_aqicn = spark.createDataFrame(pdf_aq)
print(f"df_aqicn registros: {df_aqicn.count()}")
df_aqicn.show(5)

### 1.4 — Fuente MongoDB (BPM de usuarios)
Si la conexión a Mongo falla momentáneamente, se carga un **dataset mock de respaldo** con 20 registros que incluye `alturaft` (altura en pies) requerido por la transformación posterior.

In [ ]:
from pymongo import MongoClient
import pandas as pd

MONGO_URI = "<PLACEHOLDER_MONGO_URI>"
MONGO_DB = "grupo2"
MONGO_COL = "bpm"

try:
    cli = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    docs = list(cli[MONGO_DB][MONGO_COL].find({}, {"_id": 0}).limit(20))
    if not docs:
        raise RuntimeError("coleccion vacia")
    print("[OK] MongoDB conectado.")
except Exception as e:
    print(f"[FALLBACK MONGO] {e}")
    docs = [{"usuario": f"u{i:02d}", "bpm": 60 + (i % 40), "alturaft": 5.5 + (i % 5) * 0.1} for i in range(20)]

pdf_bpm = pd.DataFrame(docs)
pdf_bpm["usuario"]  = pdf_bpm["usuario"].astype(str)
pdf_bpm["bpm"]      = pdf_bpm["bpm"].astype(float)
pdf_bpm["alturaft"] = pdf_bpm["alturaft"].astype(float)

df_mongo_bpm = spark.createDataFrame(pdf_bpm)
print(f"df_mongo_bpm registros: {df_mongo_bpm.count()}")
df_mongo_bpm.show(5)

---
## Fase 2 — Preprocesamiento y Transformación (RA4, ID4.2)

### 2.1 — Columna calculada `altura_metros` en `df_mongo_bpm`
Conversión `alturaft × 0.3048 = altura_metros`.

In [ ]:
from pyspark.sql.functions import col, round as spark_round

df_mongo_bpm = df_mongo_bpm.withColumn(
    "altura_metros", spark_round(col("alturaft") * 0.3048, 2)
)
df_mongo_bpm.show(5)

### 2.2 — Columna de clasificación `condicion_clima` en `df_openmeteo`
Lógica condicional: **CALOR Y HUMEDAD** / **CALOR SECO** / **CLIMA MODERADO** / **CONDICION DESCONOCIDA**.

In [ ]:
from pyspark.sql.functions import when, col

df_openmeteo = df_openmeteo.withColumn(
    "condicion_clima",
    when((col("temperatura_2m") > 30) & (col("humedad_relativa_2m") > 70), "CALOR Y HUMEDAD")
    .when((col("temperatura_2m") > 30) & (col("humedad_relativa_2m") <= 70), "CALOR SECO")
    .when(col("temperatura_2m") <= 30, "CLIMA MODERADO")
    .otherwise("CONDICION DESCONOCIDA")
)
df_openmeteo.select("fecha","temperatura_2m","humedad_relativa_2m","condicion_clima").show(5, truncate=False)

---
## Fase 3 — Vistas Temporales Spark SQL
Cada uno de los 4 DataFrames expone su vista temporal (requisito de rúbrica para `spark.sql(...)`).

In [ ]:
df_openmeteo.createOrReplaceTempView("tabla_openmeteo")
df_aqicn.createOrReplaceTempView("tabla_aqicn")
df_mysql_dht.createOrReplaceTempView("tabla_mysql_dht")
df_mongo_bpm.createOrReplaceTempView("tabla_mongo_bpm")
print("Vistas registradas: tabla_openmeteo, tabla_aqicn, tabla_mysql_dht, tabla_mongo_bpm")

---
## Fase 4 — Integración de Datos (RA3) — `df_unificado`

### 🧠 Justificación de la Estrategia de Integración

Las cuatro fuentes son **estructural y semánticamente heterogéneas**:

| Fuente | Tipo | Granularidad | Clave natural |
|---|---|---|---|
| `df_openmeteo` | API REST (JSON) | Hora del día (time-series) | timestamp |
| `df_aqicn` | API REST (JSON) | Por ciudad / snapshot | ciudad |
| `df_mysql_dht` | RDBMS (tabular) | Lectura física por dispositivo | (dispositivo, fecha) |
| `df_mongo_bpm` | NoSQL documental | Por usuario | usuario_id |

No existe una **clave foránea común** entre las cuatro fuentes (clima ≠ contaminación ≠ lectura IoT ≠ usuario), por lo que un `JOIN` clásico por clave de negocio no es viable y produciría producto cartesiano o pérdida de filas.

**Estrategia adoptada:** asignar un **índice secuencial sintético `id`** mediante `monotonically_increasing_id()` sobre cada DataFrame previamente forzado a una sola partición con `coalesce(1)`, y aplicar `FULL OUTER JOIN` por ese índice. Esto:

1. **Conserva los 20 registros** de cada fuente (ningún descarte).
2. **Alinea posicionalmente** los registros, asumiendo que las observaciones del mismo `id` son contemporáneas (snapshot del pipeline).
3. **Evita explosión cartesiana** que ocurriría con un cross join (20×20×20×20 = 160.000 filas).
4. **Habilita consultas cruzadas** posteriores (clima exterior + contaminación + IoT interior + usuario) en un único DataFrame ancho.

Es la estrategia recomendada en literatura Big Data para **"data fusion" de fuentes desacopladas** cuando se busca un dataset analítico unificado y no una integración transaccional.

In [ ]:
from pyspark.sql.functions import monotonically_increasing_id

def with_id(df):
    return df.coalesce(1).withColumn("id", monotonically_increasing_id())

om = with_id(df_openmeteo)
aq = with_id(df_aqicn)
my = with_id(df_mysql_dht.limit(20))
mb = with_id(df_mongo_bpm)

df_unificado = (om
    .join(aq, "id", "full_outer")
    .join(my, "id", "full_outer")
    .join(mb, "id", "full_outer"))

df_unificado.createOrReplaceTempView("tabla_unificada")
print(f"Registros unificados: {df_unificado.count()}")
df_unificado.show(5, truncate=False)

---
## Fase 5 — Consultas Spark SQL (C1 a C8) — ID3.1 / ID4.1

### C1 — Top 10 registros de OpenMeteo ordenados por temperatura DESC

In [ ]:
spark.sql("""
SELECT * FROM tabla_openmeteo
ORDER BY temperatura_2m DESC
LIMIT 10
""").show(truncate=False)

### C2 — Promedio de humedad relativa para un día y mes específicos
Día/mes elegidos: **08-05** (8 de mayo).

In [ ]:
spark.sql("""
SELECT AVG(humedad_relativa_2m) AS humedad_promedio
FROM tabla_openmeteo
WHERE MONTH(fecha) = 5 AND DAY(fecha) = 8
""").show()

### C3 — Cantidad de registros con precipitación > 2 mm (última semana)

In [ ]:
spark.sql("""
SELECT COUNT(*) AS registros_lluvia_fuerte
FROM tabla_openmeteo
WHERE precipitacion > 2
  AND CAST(fecha AS TIMESTAMP) >= current_timestamp() - INTERVAL 7 DAYS
""").show()

### C4 — Promedio de PM2.5, PM10, O3, NO2, SO2 (AQICN)

In [ ]:
spark.sql("""
SELECT AVG(pm25) AS pm25_avg, AVG(pm10) AS pm10_avg,
       AVG(o3)   AS o3_avg,   AVG(no2)  AS no2_avg,
       AVG(so2)  AS so2_avg
FROM tabla_aqicn
""").show()

### C5 — Cantidad de registros con NO2 > 4 (AQICN)

In [ ]:
spark.sql("""
SELECT COUNT(*) AS registros_no2_alto FROM tabla_aqicn WHERE no2 > 4
""").show()

### C6 — `df_mongo_bpm`: usuarios con altura_metros y conteo de taquicardia (BPM > 100)

In [ ]:
spark.sql("""
SELECT usuario,
       AVG(bpm)            AS bpm_promedio,
       AVG(altura_metros)  AS altura_m,
       SUM(CASE WHEN bpm > 100 THEN 1 ELSE 0 END) AS taquicardia_count
FROM tabla_mongo_bpm
GROUP BY usuario
ORDER BY bpm_promedio DESC
LIMIT 10
""").show()

### C7 — Primeros 10 registros de OpenMeteo incluyendo `condicion_clima`

In [ ]:
spark.sql("""
SELECT fecha, temperatura_2m, humedad_relativa_2m, condicion_clima
FROM tabla_openmeteo
LIMIT 10
""").show(truncate=False)

### C8 — `df_mysql_dht`: estadísticos de temperatura interior y humedad (sensor DHT22 ESP32)

In [ ]:
spark.sql("""
SELECT dispositivo,
       MIN(CAST(valor AS DOUBLE)) AS min_val,
       MAX(CAST(valor AS DOUBLE)) AS max_val,
       AVG(CAST(valor AS DOUBLE)) AS avg_val,
       COUNT(*)                   AS lecturas
FROM tabla_mysql_dht
WHERE dispositivo LIKE '%DHT22%'
GROUP BY dispositivo
""").show(truncate=False)

---
## Fase 6 — Valor Agregado IoT Real (C9 y C10)
Aprovechamos el sensor de gas **MQ135** del proyecto físico del estudiante (placa ESP32 Cheap Yellow Display) para análisis avanzado.

### C9 — Estadísticos del gas MQ135 (Local vs Remoto)
Promedio, máximo y mínimo del nivel de gas para cada nodo (sensor local en gateway y sensor remoto en nodo esclavo ESP-NOW).

In [ ]:
spark.sql("""
SELECT dispositivo,
       MIN(CAST(valor AS DOUBLE)) AS min_ppm,
       MAX(CAST(valor AS DOUBLE)) AS max_ppm,
       AVG(CAST(valor AS DOUBLE)) AS avg_ppm,
       COUNT(*)                   AS lecturas
FROM tabla_mysql_dht
WHERE dispositivo IN ('Local_MQ135', 'Remoto_MQ135')
GROUP BY dispositivo
""").show(truncate=False)

### C10 — Cruce de contaminación exterior (PM2.5 — AQICN) vs picos de gas interior (MQ135)
Consulta sobre `tabla_unificada` que correlaciona los niveles de **PM2.5 atmosférico** capturados por la API AQICN con las lecturas físicas del sensor **MQ135** del estudiante. Permite evaluar si la calidad del aire exterior se refleja en el ambiente interno monitoreado por la placa ESP32.

In [ ]:
from pyspark.sql.functions import monotonically_increasing_id, col

mq_local_id = (df_mysql_dht
    .filter(col("dispositivo") == "Local_MQ135")
    .limit(20)
    .coalesce(1)
    .withColumn("id", monotonically_increasing_id()))
mq_local_id.createOrReplaceTempView("tabla_mq135_local_id")

aq_id = df_aqicn.coalesce(1).withColumn("id", monotonically_increasing_id())
aq_id.createOrReplaceTempView("tabla_aqicn_id")

spark.sql("""
SELECT m.id,
       CAST(m.valor AS DOUBLE) AS gas_interior_ppm,
       a.pm25 AS pm25_exterior,
       CASE WHEN CAST(m.valor AS DOUBLE) > 400 AND a.pm25 > 35
              THEN 'ALERTA: gas interior alto + PM2.5 exterior alto'
            WHEN CAST(m.valor AS DOUBLE) > 400 THEN 'Pico gas interior'
            WHEN a.pm25 > 35 THEN 'PM2.5 exterior elevado'
            ELSE 'Normal' END AS evaluacion
FROM tabla_mq135_local_id m
FULL OUTER JOIN tabla_aqicn_id a ON m.id = a.id
ORDER BY m.id
""").show(truncate=False)

---
## Cierre
Pipeline distribuido completado: 4 fuentes heterogéneas ingestadas con 20 registros c/u, transformaciones aplicadas (`altura_metros`, `condicion_clima`), DataFrame unificado por índice sintético, 8 consultas obligatorias resueltas más 2 consultas de valor agregado sobre el sensor MQ135 real del proyecto IoT.